# 第三课：共享 MLP 与完整训练循环

前两课分别回答了“样本是什么”和“预测好坏怎样衡量”。这一课第一次训练神经网络，目标是理解：

1. 参数、梯度和优化器分别是什么；
2. 为什么共享 MLP 的参数量不随传感器数量变化；
3. `[B,T,N,1]` 如何变成每个节点一条长度为 12 的序列；
4. 一个可靠训练循环需要哪些步骤；
5. 训练损失、验证指标和测试指标为什么不能混用；
6. MLP 与 Persistence、HA、STGCN 的差异意味着什么。

> 请逐格运行。先完成小数据实验，再决定是否运行完整训练。

## 0. 环境与数据

In [ ]:
from pathlib import Path
import copy
import json
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
import torch
from torch.utils.data import DataLoader, Subset

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').exists(), '请从项目根目录或 notebooks 目录启动 notebook'
sys.path.insert(0, str(ROOT / 'src'))

from crosscity.config import load_config  # noqa: E402
from crosscity.data.dataset import build_data_bundle  # noqa: E402
from crosscity.evaluation.metrics import horizon_metrics, masked_mae  # noqa: E402
from crosscity.models import SharedMLP  # noqa: E402
from crosscity.utils import choose_device, seed_everything  # noqa: E402

seed_everything(42)
config = load_config(ROOT / 'configs/metr_la.yaml')
bundle = build_data_bundle(config.dataset)
device = choose_device()
print('device:', device)
print('train/val/test:', len(bundle.train), len(bundle.val), len(bundle.test))

如果服务器正确识别 NVIDIA GPU，`device` 应为 `cuda`。本课的 MLP 很小，CPU 也可以运行，只是完整训练更慢。

## 1. 一个参数是怎样被更新的

先忘记 207 个传感器。假设只有一个参数 $w$，模型为：

$$\hat y=wx$$

给定 $x=2,y=10$，理想参数是 $w=5$。现在从 $w=1$ 开始。

**先思考：** 当前预测是多少？要让预测接近 10，$w$ 应该增大还是减小？

In [ ]:
w = torch.tensor(1.0, requires_grad=True)
x_scalar = torch.tensor(2.0)
y_scalar = torch.tensor(10.0)

prediction_scalar = w * x_scalar
loss_scalar = (prediction_scalar - y_scalar).square()
loss_scalar.backward()

print('prediction:', prediction_scalar.item())
print('loss:', loss_scalar.item())
print('d(loss)/d(w):', w.grad.item())

梯度为负，表示增大 $w$ 会减小损失。梯度下降更新公式是：

$$w\leftarrow w-\eta\frac{\partial L}{\partial w}$$

其中 $\eta$ 是学习率。负梯度被减去，所以参数会增大。

In [ ]:
learning_rate = 0.1
with torch.no_grad():
    w -= learning_rate * w.grad
w.grad.zero_()
print('更新后的 w:', w.item())
print('更新后的预测:', (w * x_scalar).item())

真实神经网络包含许多参数，因此我们不会手工更新每一个参数。`loss.backward()` 计算梯度，`optimizer.step()` 统一更新参数，`optimizer.zero_grad()` 清除上一轮梯度。

## 2. MLP 在本项目中做什么

对每个传感器，MLP 接收过去 12 个速度，输出未来 12 个速度：

$$\mathbb{R}^{12}\rightarrow\mathbb{R}^{H}\rightarrow\mathbb{R}^{12}$$

第一层把长度 12 的历史映射到隐藏表示，ReLU 提供非线性，第二层生成 12 步预测。所有传感器使用同一套参数。

In [ ]:
model = SharedMLP(input_steps=12, output_steps=12, hidden_dim=32)
print(model)
print('参数形状：')
for name, parameter in model.named_parameters():
    print(f'{name:20s}', tuple(parameter.shape), '数量=', parameter.numel())
print('总参数量:', sum(p.numel() for p in model.parameters()))

第一层权重是 `[32,12]`，第二层是 `[12,32]`。形状中没有 207，因此参数量与城市节点数无关。

**先思考：** 如果换成 325 个传感器的 PEMS-BAY，模型参数量会变化吗？一次前向计算量会变化吗？

答案是：参数量不变，但需要处理更多节点，所以计算量会增加。

## 3. 跟踪一次前向传播的形状

In [ ]:
loader = DataLoader(bundle.train, batch_size=4, shuffle=False)
batch_x, batch_y, batch_mask = next(iter(loader))
print('原始 batch_x:', tuple(batch_x.shape), '= [B,T,N,F]')

node_sequences = batch_x[..., 0].transpose(1, 2)
print('去掉 F 并转置:', tuple(node_sequences.shape), '= [B,N,T]')

hidden = model.network[0](node_sequences)
hidden_after_relu = model.network[1](hidden)
node_predictions = model.network[2](hidden_after_relu)
final_prediction = node_predictions.transpose(1, 2)
print('隐藏表示:', tuple(hidden.shape), '= [B,N,H]')
print('节点预测:', tuple(node_predictions.shape), '= [B,N,T_out]')
print('最终输出:', tuple(final_prediction.shape), '= [B,T_out,N]')
assert torch.allclose(final_prediction, model(batch_x))

PyTorch 的 `Linear` 会作用在最后一维，所以 `[B,N,12]` 中的每个 `[12]` 都经过同一个 MLP。这就是“node-shared”。

它没有使用邻接矩阵：传感器 0 的预测只看传感器 0 的历史，不能读取传感器 1 的速度。

## 4. 一次标准训练步骤

训练循环的核心顺序必须清楚：

1. 清除旧梯度；
2. 前向传播；
3. 只在 `mask=True` 位置计算损失；
4. 反向传播；
5. 更新参数。

In [ ]:
seed_everything(42)
one_step_model = SharedMLP(12, 12, hidden_dim=32)
optimizer = torch.optim.Adam(one_step_model.parameters(), lr=1e-3)

optimizer.zero_grad()
prediction = one_step_model(batch_x)
loss = masked_mae(prediction, batch_y, batch_mask)
loss.backward()

print('标准化空间 MAE loss:', loss.item())
print('第一层梯度范数:', one_step_model.network[0].weight.grad.norm().item())

before = one_step_model.network[0].weight.detach().clone()
optimizer.step()
after = one_step_model.network[0].weight.detach().clone()
print('参数是否改变:', not torch.equal(before, after))

这里的 loss 位于标准化空间，用于稳定优化；最终报告指标时必须逆标准化。一次更新后 loss 不保证立即下降，训练关注多个 batch 和 epoch 的总体趋势。

## 5. 为什么先尝试过拟合一个 batch

在正式训练前，先让模型反复学习同一小批数据。如果连固定样本都无法拟合，通常意味着：形状、损失、梯度、学习率或模型实现存在错误。

这不是测试泛化能力，而是检查训练管线是否接通。

In [ ]:
seed_everything(42)
overfit_model = SharedMLP(12, 12, hidden_dim=64).to(device)
overfit_optimizer = torch.optim.Adam(overfit_model.parameters(), lr=0.01)
small_x = batch_x[:1].to(device)
small_y = batch_y[:1].to(device)
small_mask = batch_mask[:1].to(device)
overfit_losses = []

for step in range(201):
    overfit_optimizer.zero_grad()
    small_prediction = overfit_model(small_x)
    small_loss = masked_mae(small_prediction, small_y, small_mask)
    small_loss.backward()
    overfit_optimizer.step()
    overfit_losses.append(small_loss.item())

print('initial loss:', overfit_losses[0])
print('final loss  :', overfit_losses[-1])

In [ ]:
plt.figure(figsize=(7, 3))
plt.plot(overfit_losses)
plt.xlabel('Optimization step')
plt.ylabel('Masked MAE in scaled space')
plt.title('Overfitting one fixed batch')
plt.yscale('log')
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()

如果曲线整体明显下降，说明模型、损失和优化器能够协同工作。它不需要严格单调，因为 Adam 的更新和 MAE 的折点可能造成小幅波动。

## 6. Epoch、batch 和 shuffle

- batch：一次参数更新所使用的一组样本；
- epoch：模型完整看过一次训练集；
- shuffle：每个 epoch 改变训练样本顺序，降低顺序相关的优化偏差。

验证集不能 shuffle 也不更新参数。训练模式使用 `model.train()`；验证模式使用 `model.eval()` 和 `torch.no_grad()`。MLP 当前没有 Dropout/BatchNorm，但养成正确习惯很重要。

## 7. 写出完整训练与验证函数

In [ ]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    loss_sum = 0.0
    sample_count = 0
    for x, y, mask in loader:
        x, y, mask = x.to(device), y.to(device), mask.to(device)
        optimizer.zero_grad()
        prediction = model(x)
        loss = masked_mae(prediction, y, mask)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        loss_sum += loss.item() * x.shape[0]
        sample_count += x.shape[0]
    return loss_sum / sample_count


@torch.no_grad()
def evaluate_scaled_mae(model, loader, device):
    model.eval()
    predictions, targets, masks = [], [], []
    for x, y, mask in loader:
        predictions.append(model(x.to(device)).cpu())
        targets.append(y)
        masks.append(mask)
    prediction = torch.cat(predictions)
    target = torch.cat(targets)
    mask = torch.cat(masks)
    return masked_mae(prediction, target, mask).item()

请逐行确认：验证函数没有 `backward()` 和 `optimizer.step()`。测试集在训练过程中甚至不应传入这些函数。

## 8. 先做快速训练

默认使用前 4096 个训练窗口、前 1024 个验证窗口和 8 个 epoch，几分钟内验证趋势。这里取连续子集只是为了教学速度，不能作为正式论文结果。

In [ ]:
FAST_MODE = True
if FAST_MODE:
    train_dataset = Subset(bundle.train, range(min(4096, len(bundle.train))))
    val_dataset = Subset(bundle.val, range(min(1024, len(bundle.val))))
    max_epochs = 8
else:
    train_dataset = bundle.train
    val_dataset = bundle.val
    max_epochs = 30

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
print('mode:', 'FAST' if FAST_MODE else 'FULL')
print('train/val windows:', len(train_dataset), len(val_dataset), 'epochs:', max_epochs)

In [ ]:
seed_everything(42)
mlp = SharedMLP(12, 12, hidden_dim=32).to(device)
optimizer = torch.optim.Adam(mlp.parameters(), lr=1e-3, weight_decay=1e-4)
history = []
best_val = float('inf')
best_state = None
patience = 5
stale_epochs = 0
start_time = time.time()

for epoch in range(1, max_epochs + 1):
    train_loss = train_one_epoch(mlp, train_loader, optimizer, device)
    val_loss = evaluate_scaled_mae(mlp, val_loader, device)
    history.append({'epoch': epoch, 'train_scaled_mae': train_loss, 'val_scaled_mae': val_loss})
    print(f'epoch={epoch:02d} train={train_loss:.4f} val={val_loss:.4f}')
    if val_loss < best_val:
        best_val = val_loss
        best_state = copy.deepcopy(mlp.state_dict())
        stale_epochs = 0
    else:
        stale_epochs += 1
    if stale_epochs >= patience:
        print('early stopping')
        break

mlp.load_state_dict(best_state)
print(f'elapsed: {time.time() - start_time:.1f}s, best val={best_val:.4f}')

In [ ]:
history_frame = pd.DataFrame(history)
display(history_frame)
plt.figure(figsize=(7, 3.5))
plt.plot(history_frame['epoch'], history_frame['train_scaled_mae'], marker='o', label='train')
plt.plot(history_frame['epoch'], history_frame['val_scaled_mae'], marker='o', label='validation')
plt.xlabel('Epoch')
plt.ylabel('Masked MAE (scaled space)')
plt.title('Shared MLP learning curve')
plt.grid(alpha=0.2)
plt.legend()
plt.tight_layout()
plt.show()

读曲线时检查：

- train 和 validation 都下降：模型正在学习可泛化规律；
- train 下降但 validation 上升：开始过拟合；
- 二者几乎不下降：检查学习率、梯度、模型容量或数据；
- validation 比 train 略低不一定是错误，两者的数据难度可能不同。

## 9. 测试集只在模型选择结束后使用

下面第一次评估测试集。将预测和目标逆标准化，报告真实速度单位。

In [ ]:
@torch.no_grad()
def predict_dataset(model, dataset, device, batch_size=64):
    model.eval()
    predictions, targets, masks = [], [], []
    for x, y, mask in DataLoader(dataset, batch_size=batch_size, shuffle=False):
        predictions.append(model(x.to(device)).cpu())
        targets.append(y)
        masks.append(mask)
    return torch.cat(predictions), torch.cat(targets), torch.cat(masks)

scaled_prediction, scaled_target, test_mask = predict_dataset(mlp, bundle.test, device)
prediction_speed = bundle.scaler.inverse_transform(scaled_prediction)
target_speed = bundle.scaler.inverse_transform(scaled_target)
mlp_metrics = horizon_metrics(prediction_speed, target_speed, test_mask)
pd.DataFrame(mlp_metrics).T

如果 `FAST_MODE=True`，这些只是教学 smoke-test 指标，不应与完整训练的 STGCN 做严格结论。要得到正式 MLP 基线，将 `FAST_MODE=False`，从创建 loader 的单元格重新运行到这里。

## 10. 与上一课结果比较

填写上一课得到的 Persistence 和 HA 数值。下面已经放入你报告的结果。

In [ ]:
previous_mae = {
    'Historical Average': {'overall': 4.188, 'horizon_3': 4.188, 'horizon_6': 4.188, 'horizon_12': 4.189},
    'Persistence': {'overall': 4.284, 'horizon_3': 3.531, 'horizon_6': 4.252, 'horizon_12': 5.429},
    'STGCN': {'overall': 3.887, 'horizon_3': 3.286, 'horizon_6': 3.857, 'horizon_12': 4.786},
    'Shared MLP (this run)': {h: values['mae'] for h, values in mlp_metrics.items()},
}
comparison = pd.DataFrame(previous_mae).loc[['horizon_3', 'horizon_6', 'horizon_12', 'overall']]
comparison.round(3)

分析时区分三个问题：

1. MLP 是否战胜无需训练的基线？这检查非线性历史映射是否有价值。
2. STGCN 是否战胜 MLP？这初步检查空间图信息是否有价值。
3. 哪个模型在 60 分钟最好？远期预测可能更依赖周期信息。

只有在 MLP 完整训练后，第 1、2 项才是公平比较。

## 11. 保存本次教学实验（可选）

只有在 `FAST_MODE=False` 时才建议保存为正式 MLP 结果。

In [ ]:
if not FAST_MODE:
    output_dir = ROOT / 'artifacts/metr_la/mlp/target-full/seed-42'
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save({'model_state': mlp.state_dict(), 'best_val_scaled_mae': best_val}, output_dir / 'best.pt')
    (output_dir / 'history.json').write_text(json.dumps(history, indent=2), encoding='utf-8')
    payload = {
        'metadata': {'city': 'metr_la', 'model': 'mlp', 'mode': 'target-full', 'seed': 42},
        'metrics': mlp_metrics,
    }
    (output_dir / 'metrics.json').write_text(json.dumps(payload, indent=2), encoding='utf-8')
    print('saved to', output_dir)
else:
    print('FAST_MODE result not saved as a formal experiment')

## 12. 本课验收

请不用查看上文回答：

1. `loss.backward()`、`optimizer.step()`、`optimizer.zero_grad()` 分别做什么？
2. 为什么 SharedMLP 从 207 个节点换到 325 个节点时参数量不变？
3. MLP 为什么不需要 adjacency？这同时造成什么能力限制？
4. 为什么要先尝试过拟合一个固定 batch？
5. epoch 和 batch 有什么区别？
6. 为什么根据 validation 选择 checkpoint，而不能根据 test 选择？
7. 什么曲线特征表示可能发生了过拟合？
8. 为什么 FAST_MODE 的结果不能当作正式基线？

运行模型和训练链路测试：

In [ ]:
import subprocess

result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', 'tests/test_models.py', 'tests/test_training.py'],
    cwd=ROOT, text=True, capture_output=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr)
assert result.returncode == 0, '训练链路测试未通过，请把错误发给我'

## 13. 请带回来的结果

请先完成 FAST_MODE，再根据时间运行 FULL_MODE。记录：

- 单 batch 过拟合的 initial/final loss；
- FAST_MODE 每轮 train/validation loss；
- FULL_MODE 最佳 epoch 和测试集四个 MAE；
- MLP、Persistence、HA、STGCN 各 horizon 的排名；
- 训练曲线是否出现过拟合；
- 八个验收问题中最不确定的一项。

下一课将把 MLP 的一次性时间映射替换为 LSTM 的逐时刻状态更新，理解循环网络怎样表示速度变化过程。